In [1]:
# 1. Prerequisites

In [2]:
import sys
from pathlib import Path

# Make local Final_Task utilities importable even when Jupyter starts outside this folder.
final_task_dir = Path.cwd()
if not (final_task_dir / "utils").exists():
    candidates = [
        Path.home() / "pands_ibnnwml" / "Final_Task",
        Path.cwd() / "pands_ibnnwml" / "Final_Task",
        Path.cwd().parent / "Final_Task",
    ]
    final_task_dir = next((path for path in candidates if (path / "utils").exists()), final_task_dir)

if str(final_task_dir) not in sys.path:
    sys.path.insert(0, str(final_task_dir))

from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import getpass
import json
import pandas as pd
from datetime import datetime

In [4]:
# Load the specific network here
Network = 5
DIV     = 21
group_data  = True
test_data = False

In [5]:
data = load_data(Network, DIV, group_data)
stimulation_parameters, stimulation_patterns, binned_spike_train_responses, stimulation_times, impedance_map, electrodes = data
print(f"Loaded data with response shape {binned_spike_train_responses.shape} and parameter shape {stimulation_parameters.shape}.")
if test_data:
    data = load_data(Network, DIV, group_data, test_mode=True)
    stimulation_parameters_test, stimulation_patterns_test, binned_spike_train_responses_test, stimulation_times_test, _, _ = data
else:
    binned_spike_train_responses_test = stimulation_patterns_test = stimulation_parameters_test = stimulation_times_test = None

Loaded data with response shape (35640, 160, 80) and parameter shape (35640, 2).


In [6]:
# 2. Model Definitions and Data Preparation

In [7]:
class GraphBlock(nn.Module):
    def __init__(self, hidden_dim, dropout=0.20):
        super().__init__()
        self.update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h, adj):
        neigh = torch.einsum("ij,bjh->bih", adj, h)
        delta = self.update(torch.cat([h, neigh], dim=-1))
        return self.norm(h + delta)


class CNN(nn.Module):
    """
    Coordinate-aware graph temporal classifier for Task 1.

    Uses the full available response structure without using the target label as input:
    - temporal filters per electrode
    - physical electrode coordinates
    - train-only electrode rate feature
    - learned electrode identity embedding
    - kNN graph message passing over the electrode geometry
    - known current frequency as context
    """

    def __init__(
        self,
        n_outputs=16,
        n_electrodes=160,
        coords_norm=None,
        adjacency=None,
        electrode_rate=None,
        electrode_impedance=None,
        class_prototypes=None,
        n_base_channels=40,
        hidden_dim=96,
        electrode_emb_dim=24,
        dropout=0.50,
    ):
        super().__init__()
        self.n_electrodes = n_electrodes
        self.hidden_dim = hidden_dim

        if coords_norm is None:
            coords_norm = np.stack([np.linspace(-1, 1, n_electrodes), np.zeros(n_electrodes)], axis=1).astype(np.float32)
        if adjacency is None:
            adjacency = np.eye(n_electrodes, dtype=np.float32)
        if electrode_rate is None:
            electrode_rate = np.zeros(n_electrodes, dtype=np.float32)
        if electrode_impedance is None:
            electrode_impedance = np.zeros(n_electrodes, dtype=np.float32)
        electrode_index = np.linspace(-1.0, 1.0, n_electrodes, dtype=np.float32)
        electrode_static = np.stack([electrode_rate, electrode_impedance, electrode_index], axis=1).astype(np.float32)

        self.register_buffer("coords", torch.as_tensor(coords_norm, dtype=torch.float32))
        self.register_buffer("adj", torch.as_tensor(adjacency, dtype=torch.float32))
        self.register_buffer("electrode_static", torch.as_tensor(electrode_static, dtype=torch.float32))
        if class_prototypes is None:
            class_prototypes = torch.empty(0, dtype=torch.float32)
        class_prototypes = torch.as_tensor(class_prototypes, dtype=torch.float32)
        if class_prototypes.numel() > 0:
            class_prototypes = F.normalize(class_prototypes.flatten(start_dim=1), dim=1)
        self.register_buffer("class_prototypes", class_prototypes)
        self.prototype_scale = nn.Parameter(torch.tensor(5.0))

        self.temporal_layers = nn.Sequential(
            nn.Conv2d(1, n_base_channels, kernel_size=(1, 9), padding=(0, 4), bias=False),
            nn.BatchNorm2d(n_base_channels),
            nn.SiLU(),
            nn.Dropout2d(0.05),
            nn.Conv2d(n_base_channels, n_base_channels, kernel_size=(1, 7), padding=(0, 3), bias=False),
            nn.BatchNorm2d(n_base_channels),
            nn.SiLU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Conv2d(n_base_channels, n_base_channels * 2, kernel_size=(1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(n_base_channels * 2),
            nn.SiLU(),
            nn.Dropout2d(0.10),
            nn.Conv2d(n_base_channels * 2, n_base_channels * 2, kernel_size=(1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(n_base_channels * 2),
            nn.SiLU(),
        )

        temporal_feature_dim = n_base_channels * 2 * 2  # avg + max over time
        electrode_feature_dim = temporal_feature_dim + 2 + 3 + electrode_emb_dim
        self.electrode_emb = nn.Embedding(n_electrodes, electrode_emb_dim)
        self.electrode_mlp = nn.Sequential(
            nn.Linear(electrode_feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.35),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.graph_blocks = nn.ModuleList([GraphBlock(hidden_dim, dropout=dropout * 0.35) for _ in range(2)])

        freq_dim = 5
        pooled_dim = hidden_dim * n_electrodes + hidden_dim * 2 + freq_dim
        self.classifier_features = nn.Sequential(
            nn.Linear(pooled_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 192),
            nn.BatchNorm1d(192),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.class_head = nn.Linear(192, n_outputs)
        self.bit_head = nn.Linear(192, 4)

    def frequency_features(self, freq):
        freq = freq.view(-1, 1).to(dtype=self.coords.dtype)
        return torch.cat([
            freq,
            freq.square(),
            torch.sin(np.pi * freq),
            torch.cos(np.pi * freq),
            torch.sin(2 * np.pi * freq),
        ], dim=1)

    def forward(self, x, freq=None):
        b = x.shape[0]
        raw_x = x
        x = self.temporal_layers(x)
        avg_t = x.mean(dim=-1).transpose(1, 2)  # [B, N, C]
        max_t = x.amax(dim=-1).transpose(1, 2)
        temporal = torch.cat([avg_t, max_t], dim=-1)

        ids = torch.arange(self.n_electrodes, device=x.device)
        coord = self.coords.to(x.device)[None, :, :].expand(b, -1, -1)
        static = self.electrode_static.to(x.device)[None, :, :].expand(b, -1, -1)
        emb = self.electrode_emb(ids)[None, :, :].expand(b, -1, -1)
        h = self.electrode_mlp(torch.cat([temporal, coord, static, emb], dim=-1))

        adj = self.adj.to(x.device)
        for block in self.graph_blocks:
            h = block(h, adj)

        flat = torch.flatten(h, start_dim=1)
        mean_pool = h.mean(dim=1)
        max_pool = h.amax(dim=1)

        if freq is None:
            freq = torch.zeros(b, device=x.device, dtype=x.dtype)
        else:
            freq = freq.to(device=x.device, dtype=x.dtype)
        freq_feat = self.frequency_features(freq)

        features = torch.cat([flat, mean_pool, max_pool, freq_feat], dim=1)
        cls_features = self.classifier_features(features)
        class_logits = self.class_head(cls_features)
        bit_logits = self.bit_head(cls_features)

        if self.class_prototypes.numel() > 0:
            raw_flat = F.normalize(raw_x.flatten(start_dim=1), dim=1)
            proto_logits = raw_flat @ self.class_prototypes.to(raw_x.device).T
            class_logits = class_logits + self.prototype_scale * proto_logits

        return class_logits, bit_logits

In [8]:
# Hyperparameters
# More epochs + a stronger optimizer give the larger CNN enough room to improve.
n_epochs = 120
batch_size = 384
learning_rate = 8e-4
weight_decay = 5e-4

# Integer class targets are the standard input for CrossEntropyLoss.
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

n_classes = 16 # This is given by the data

# A CNN expects input shape (Batch, Channel, Height, Width). Add the channel dimension once.
if binned_spike_train_responses.ndim != 4:
    print(f"Initial input data size is {binned_spike_train_responses.shape}")
    binned_spike_train_responses = binned_spike_train_responses[:, None]
    print(f"Adjusted input data size is {binned_spike_train_responses.shape}")
if binned_spike_train_responses_test is not None and binned_spike_train_responses_test.ndim != 4:
    binned_spike_train_responses_test = binned_spike_train_responses_test[:, None]

print("Model, optimizer, and scheduler will be initialized after train-only electrode features are computed.")


Initial input data size is (35640, 160, 80)
Adjusted input data size is (35640, 1, 160, 80)
Model, optimizer, and scheduler will be initialized after train-only electrode features are computed.


In [9]:
# Data loading
# Pytorch dataloaders are parallel processing classes optimized for machine learning with Python.
# We split the training data into train and validation sets.

val_frac = 0.1
rng = np.random.default_rng(42)

patterns_np = np.asarray(stimulation_patterns).astype(np.int64)
frequencies_np = np.asarray(stimulation_parameters)[:, 0]

# Stratify by frequency-pattern pairs so validation accuracy is less dependent on one lucky split.
strata = np.array([f"{freq}_{pat}" for freq, pat in zip(frequencies_np, patterns_np)])
val_indices = []
train_indices = []

for stratum in np.unique(strata):
    stratum_idx = np.flatnonzero(strata == stratum)
    rng.shuffle(stratum_idx)
    n_stratum_val = max(1, int(round(len(stratum_idx) * val_frac)))
    val_indices.append(stratum_idx[:n_stratum_val])
    train_indices.append(stratum_idx[n_stratum_val:])

val_idx = torch.as_tensor(np.concatenate(val_indices), dtype=torch.long)
trn_idx = torch.as_tensor(np.concatenate(train_indices), dtype=torch.long)

# Shuffle after stratification so batches are mixed.
val_idx = val_idx[torch.randperm(len(val_idx))]
trn_idx = trn_idx[torch.randperm(len(trn_idx))]

# Here, Z is any additional information we want to keep with the dataset split.
Xtr, Ytr, Ztr = (
    binned_spike_train_responses[trn_idx],
    patterns_np[trn_idx.numpy()],
    stimulation_parameters[trn_idx],
)

Xval, Yval, Zval = (
    binned_spike_train_responses[val_idx],
    patterns_np[val_idx.numpy()],
    stimulation_parameters[val_idx],
)

# Convert inputs and labels to tensors.
Xtr = torch.as_tensor(Xtr, dtype=torch.float32)
Xval = torch.as_tensor(Xval, dtype=torch.float32)
Ytr = torch.as_tensor(Ytr, dtype=torch.long)
Yval = torch.as_tensor(Yval, dtype=torch.long)

# Reduce skew in spike counts.
Xtr = torch.log1p(Xtr)
Xval = torch.log1p(Xval)

# Train-only electrode rate feature: computed after log1p but before global response normalization.
electrode_rate = Xtr.mean(dim=(0, 1, 3)).numpy().astype(np.float32)
electrode_rate = (electrode_rate - electrode_rate.mean()) / (electrode_rate.std() + 1e-6)

# Normalize using TRAINING data statistics only.
mean = Xtr.mean()
std = Xtr.std() + 1e-6

Xtr = (Xtr - mean) / std
Xval = (Xval - mean) / std

bit_positions = torch.arange(4, dtype=torch.long)
Ytr_bits = ((Ytr[:, None] >> bit_positions[None, :]) & 1).float()
Yval_bits = ((Yval[:, None] >> bit_positions[None, :]) & 1).float()

class_prototypes = []
for class_id in range(n_classes):
    mask = Ytr == class_id
    if bool(mask.any()):
        class_prototypes.append(Xtr[mask].mean(dim=0))
    else:
        class_prototypes.append(torch.zeros_like(Xtr[0]))
class_prototypes = torch.stack(class_prototypes, dim=0)

train_dataset = TensorDataset(
    Xtr,
    Ytr,
    Ytr_bits,
    torch.as_tensor(Ztr, dtype=torch.float32)
)

val_dataset = TensorDataset(
    Xval,
    Yval,
    Yval_bits,
    torch.as_tensor(Zval, dtype=torch.float32)
)

# Keep validation labels available for benchmarking cells after freeing tensors.
val_true_labels = Yval.numpy()
val_frequencies = np.asarray(Zval)[:, 0].astype(np.float32)
freq_mean = float(np.asarray(Ztr)[:, 0].mean())
freq_std = float(np.asarray(Ztr)[:, 0].std() + 1e-6)
print(f"Frequency normalization: mean={freq_mean:.3f}, std={freq_std:.3f}")


def extract_electrode_xy(electrodes, impedance_map):
    electrodes_arr = np.asarray(electrodes)
    if electrodes_arr.ndim == 2 and electrodes_arr.shape[1] >= 2:
        return electrodes_arr[:, :2].astype(np.float32)

    imp = np.squeeze(np.asarray(impedance_map))
    if imp.ndim != 2:
        raise RuntimeError(f"Cannot infer coordinates from impedance_map with shape {imp.shape}")

    width = imp.shape[1]
    coords = np.stack([electrodes_arr % width, electrodes_arr // width], axis=1).astype(np.float32)
    if coords.shape != (len(electrodes_arr), 2):
        raise RuntimeError(f"Invalid coordinate shape {coords.shape}")
    if np.unique(coords, axis=0).shape[0] < max(5, len(electrodes_arr) // 4):
        raise RuntimeError("Degenerate electrode coordinates; too many duplicate positions.")
    return coords


def build_knn_adjacency(coords_norm, k=8, self_weight=1.0):
    coords_arr = np.asarray(coords_norm, dtype=np.float32)
    diff = coords_arr[:, None, :] - coords_arr[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=-1))
    np.fill_diagonal(dist, np.inf)
    nn_idx = np.argsort(dist, axis=1)[:, :k]
    adj = np.zeros((coords_arr.shape[0], coords_arr.shape[0]), dtype=np.float32)
    rows = np.arange(coords_arr.shape[0])[:, None]
    adj[rows, nn_idx] = 1.0
    adj = np.maximum(adj, adj.T)
    np.fill_diagonal(adj, self_weight)
    adj = adj / (adj.sum(axis=1, keepdims=True) + 1e-6)
    return adj.astype(np.float32)


coords = extract_electrode_xy(electrodes, impedance_map)
imp = np.squeeze(np.asarray(impedance_map))
if imp.ndim == 2:
    xs = np.clip(coords[:, 0].round().astype(int), 0, imp.shape[1] - 1)
    ys = np.clip(coords[:, 1].round().astype(int), 0, imp.shape[0] - 1)
    electrode_impedance = imp[ys, xs].astype(np.float32)
else:
    electrode_impedance = np.zeros(coords.shape[0], dtype=np.float32)
electrode_impedance = np.nan_to_num(electrode_impedance, nan=float(np.nanmean(electrode_impedance)))
electrode_impedance = (electrode_impedance - electrode_impedance.mean()) / (electrode_impedance.std() + 1e-6)

coord_mean = coords.mean(axis=0, keepdims=True)
coord_std = coords.std(axis=0, keepdims=True) + 1e-6
coords_norm = ((coords - coord_mean) / coord_std).astype(np.float32)
adjacency = build_knn_adjacency(coords_norm, k=8)

print("Electrode coordinate ranges:", (float(coords[:, 0].min()), float(coords[:, 0].max())), (float(coords[:, 1].min()), float(coords[:, 1].max())))
print("Graph adjacency shape:", adjacency.shape, "row sum range:", float(adjacency.sum(axis=1).min()), float(adjacency.sum(axis=1).max()))
print("Electrode impedance z range:", float(electrode_impedance.min()), float(electrode_impedance.max()))

model = CNN(
    n_classes,
    n_electrodes=binned_spike_train_responses.shape[-2],
    coords_norm=coords_norm,
    adjacency=adjacency,
    electrode_rate=electrode_rate,
    electrode_impedance=electrode_impedance,
    class_prototypes=class_prototypes,
    n_base_channels=40,
    hidden_dim=96,
    dropout=0.50,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=6
)
print(f"The model has {sum(p.numel() for p in model.parameters()):,} parameters.")

# Through deletion we can free up memory from our system.
del Xtr, Ytr, Ztr
del Xval, Yval
del binned_spike_train_responses, stimulation_parameters, stimulation_patterns

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

if binned_spike_train_responses_test is not None:
    Xtest = torch.as_tensor(
        binned_spike_train_responses_test,
        dtype=torch.float32
    )

    # Same preprocessing as train/validation.
    Xtest = torch.log1p(Xtest)
    Xtest = (Xtest - mean) / std

    test_dataset = TensorDataset(Xtest)

    # Don't shuffle this data, otherwise predictions will be random with respect to the ground truth.
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    del Xtest
    del binned_spike_train_responses_test, stimulation_parameters_test, stimulation_patterns_test

else:
    test_loader = None

Frequency normalization: mean=26.907, std=9.412
Electrode coordinate ranges: (105.0, 142.0) (41.0, 77.0)
Graph adjacency shape: (160, 160) row sum range: 0.9999998211860657 1.0
Electrode impedance z range: -1.500339388847351 1.8050568103790283
The model has 8,272,829 parameters.


In [10]:
# 3. Training and Validation Loop

In [ ]:
import os
import getpass
import json
import subprocess
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm


def write_task1_artifacts(
    output_dir,
    history_rows,
    summary,
    best_predictions=None,
    best_targets=None,
    best_frequencies=None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(output_dir / "training_history.csv", index=False)

    with open(output_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    if best_predictions is None or best_targets is None:
        return

    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(best_targets)),
        "frequency": best_frequencies if best_frequencies is not None else np.nan,
        "true_pattern": best_targets.astype(int),
        "pred_pattern": best_predictions.astype(int),
        "correct": (best_predictions == best_targets).astype(int),
    })
    pred_df.to_csv(output_dir / "best_validation_predictions.csv", index=False)

    pattern_rows = []
    for pattern in np.unique(best_targets):
        mask = best_targets == pattern
        pattern_rows.append({
            "pattern": int(pattern),
            "n_samples": int(mask.sum()),
            "accuracy": float((best_predictions[mask] == best_targets[mask]).mean()),
            "accuracy_percent": float(100 * (best_predictions[mask] == best_targets[mask]).mean()),
        })
    pd.DataFrame(pattern_rows).to_csv(output_dir / "accuracy_by_pattern.csv", index=False)

    if best_frequencies is not None:
        frequency_rows = []
        for frequency in np.unique(best_frequencies):
            mask = best_frequencies == frequency
            frequency_rows.append({
                "frequency": float(frequency),
                "n_samples": int(mask.sum()),
                "accuracy": float((best_predictions[mask] == best_targets[mask]).mean()),
                "accuracy_percent": float(100 * (best_predictions[mask] == best_targets[mask]).mean()),
            })
        pd.DataFrame(frequency_rows).to_csv(output_dir / "accuracy_by_frequency.csv", index=False)

    confusion = pd.crosstab(
        pd.Series(best_targets.astype(int), name="true_pattern"),
        pd.Series(best_predictions.astype(int), name="pred_pattern"),
        dropna=False,
    )
    confusion.to_csv(output_dir / "confusion_matrix.csv")

    np.savez_compressed(
        output_dir / "best_validation_arrays.npz",
        predictions=best_predictions,
        targets=best_targets,
        frequencies=best_frequencies,
    )


def _parse_visible_cuda_devices():
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    if not visible:
        return None
    devices = [item.strip() for item in visible.split(",") if item.strip()]
    if all(item.isdigit() for item in devices):
        return [int(item) for item in devices]
    return None


def get_idle_cuda_devices(
    max_devices=2,
    max_util_percent=1,
    min_free_memory_mb=6500,
    util_samples=5,
    sample_interval_sec=1.0,
    fallback_allow_any_visible_gpu_if_nvidia_smi_fails=False,
):
    """Return logical CUDA device ids whose sampled utilization stays <= max_util_percent.

    Utilization is the main busy-signal. Free memory is still required as a
    safety headroom check, because low-util GPUs can be occupied by paused/idle
    CUDA processes and still OOM immediately.
    """
    if not torch.cuda.is_available():
        return []

    n_torch = torch.cuda.device_count()
    visible_physical_ids = _parse_visible_cuda_devices()

    def query_once():
        try:
            result = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=index,memory.used,memory.free,memory.total,utilization.gpu",
                    "--format=csv,noheader,nounits",
                ],
                check=True,
                capture_output=True,
                text=True,
            )
        except Exception as exc:
            print(f"Could not query nvidia-smi ({exc}).")
            if fallback_allow_any_visible_gpu_if_nvidia_smi_fails:
                return [
                    dict(logical_id=i, physical_id=i, memory_used_mb=0, memory_free_mb=6500, memory_total_mb=0, util_percent=0)
                    for i in range(n_torch)
                ]
            return []

        rows = []
        for line in result.stdout.strip().splitlines():
            parts = [part.strip() for part in line.split(",")]
            if len(parts) != 5:
                continue
            physical_id, memory_used_mb, memory_free_mb, memory_total_mb, util_percent = map(int, parts)

            if visible_physical_ids is not None:
                if physical_id not in visible_physical_ids:
                    continue
                logical_id = visible_physical_ids.index(physical_id)
            else:
                logical_id = physical_id

            if logical_id >= n_torch:
                continue

            rows.append(dict(
                logical_id=int(logical_id),
                physical_id=int(physical_id),
                memory_used_mb=int(memory_used_mb),
                memory_free_mb=int(memory_free_mb),
                memory_total_mb=int(memory_total_mb),
                util_percent=int(util_percent),
            ))
        return rows

    samples = []
    for sample_idx in range(int(util_samples)):
        rows = query_once()
        if rows:
            samples.append(rows)
        if sample_idx + 1 < int(util_samples):
            time.sleep(float(sample_interval_sec))

    if not samples:
        return []

    by_logical_id = {}
    for rows in samples:
        for row in rows:
            logical_id = int(row["logical_id"])
            rec = by_logical_id.setdefault(logical_id, dict(
                logical_id=logical_id,
                physical_id=int(row["physical_id"]),
                memory_used_mb=int(row["memory_used_mb"]),
                memory_free_mb=int(row["memory_free_mb"]),
                memory_total_mb=int(row["memory_total_mb"]),
                util_samples=[],
            ))
            rec["physical_id"] = int(row["physical_id"])
            rec["memory_used_mb"] = int(row["memory_used_mb"])
            rec["memory_free_mb"] = int(row["memory_free_mb"])
            rec["memory_total_mb"] = int(row["memory_total_mb"])
            rec["util_samples"].append(int(row["util_percent"]))

    idle_logical_ids = []
    print("Visible GPU status from utilization samples:")
    for logical_id in sorted(by_logical_id):
        rec = by_logical_id[logical_id]
        util_max = max(rec["util_samples"]) if rec["util_samples"] else 0
        enough_memory = rec["memory_free_mb"] >= int(min_free_memory_mb)
        is_idle = util_max <= int(max_util_percent) and enough_memory
        tag = "free" if is_idle else ("low-util-but-memory-full" if util_max <= int(max_util_percent) else "busy")
        print(
            f" - logical GPU {logical_id} / physical GPU {rec['physical_id']}: "
            f"util_samples={rec['util_samples']}, max_util={util_max}%, "
            f"free_mem={rec['memory_free_mb']}/{rec['memory_total_mb']} MiB, "
            f"used_mem={rec['memory_used_mb']} MiB -> {tag}"
        )
        if is_idle:
            idle_logical_ids.append(logical_id)

    return idle_logical_ids[:max_devices]

task1_gpu_ids = get_idle_cuda_devices(max_devices=2)
if len(task1_gpu_ids) < 2:
    raise RuntimeError(
        "Task 1 is configured to train on two GPUs with max sampled utilization <= 1% "
        "and at least 6500 MiB free memory, "
        f"but fewer than two free GPUs were found. Free logical GPU ids found: {task1_gpu_ids}. "
        "Wait for active GPU jobs to finish or lower the requested GPU count."
    )

device = torch.device(f"cuda:{task1_gpu_ids[0]}")
model = nn.DataParallel(model, device_ids=task1_gpu_ids)
model.to(device)
print(f"Running final task model on GPUs {task1_gpu_ids} with DataParallel.")

checkpoint_model = model.module if isinstance(model, nn.DataParallel) else model

run_name = datetime.now().strftime("final_task_1_accuracy_%Y%m%d_%H%M%S")
task1_output_root = Path.cwd() / "task1_outputs"
if not (Path.cwd() / "Final_Task_1.ipynb").exists() and "final_task_dir" in globals():
    task1_output_root = Path(final_task_dir) / "task1_outputs"
task1_output_dir = task1_output_root / run_name
task1_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Task 1 output directory: {task1_output_dir}")

# Accuracy-focused checkpoint for the stronger architecture.
best_model_path = f"/home/{getpass.getuser()}/best_model_final_task_1_graph_proto_bits.pth"
run_best_model_path = task1_output_dir / "best_model_this_run.pth"

# Load previous all-time best accuracy, if available and compatible.
all_time_best_val_acc = 0.0
all_time_best_val_loss = float("inf")

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
        and checkpoint.get("model_type") == "graph_temporal_proto_bit_cnn"
    ):
        all_time_best_val_acc = checkpoint.get("best_val_acc", 0.0)
        all_time_best_val_loss = checkpoint.get("best_val_loss", float("inf"))
        print(f"Previous all-time best final-task validation accuracy: {100 * all_time_best_val_acc:.2f}%")
        print(f"Previous all-time best final-task validation loss: {all_time_best_val_loss:.4f}")
        print(f"Previous all-time best final-task epoch: {checkpoint.get('best_epoch', 'unknown')}")
        print(f"Previous all-time best final-task run: {checkpoint.get('run_name', 'unknown')}")
    else:
        print("Found incompatible or old-format Task 1 checkpoint. This architecture will start a fresh all-time best.")
else:
    print("No previous accuracy-focused final-task model found.")

pbar = tqdm(range(n_epochs), leave=False)

train_losses = np.zeros(n_epochs)
train_accuracies = np.zeros(n_epochs)
validation_losses = dict()
validation_predictions = dict()
validation_accuracies = dict()
history_rows = []

current_run_best_val_loss = float("inf")
current_run_best_loss_epoch = None
current_run_best_val_acc = 0.0
current_run_best_acc_epoch = None
current_run_best_predictions = None
current_run_best_targets = None
current_run_best_frequencies = None
completed_epochs = 0
interrupted = False

try:
    for epoch in pbar:
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for x, y, y_bits, z in train_loader:
            x, y, y_bits = x.to(device), y.to(device), y_bits.to(device)
            freq = ((z[:, 0].to(device) - freq_mean) / freq_std).float()

            optimizer.zero_grad()
            pred, bit_pred = model(x, freq)
            loss = criterion(pred, y) + 0.25 * F.binary_cross_entropy_with_logits(bit_pred, y_bits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_loss += loss.item()
            train_correct += (pred.argmax(dim=-1) == y).sum().item()
            train_total += y.numel()

        train_loss /= len(train_loader)
        train_acc = train_correct / train_total
        train_losses[epoch] = train_loss
        train_accuracies[epoch] = train_acc

        model.eval()
        val_loss = []
        val_pred = []
        val_targets = []
        val_frequencies = []

        with torch.no_grad():
            for x, y, y_bits, z in val_loader:
                x, y = x.to(device), y.to(device)
                freq = ((z[:, 0].to(device) - freq_mean) / freq_std).float()

                pred, bit_pred = model(x, freq)
                loss = criterion(pred, y)

                val_loss.append(loss.detach().cpu().numpy().reshape(1))
                val_pred.append(torch.argmax(pred, dim=-1).detach().cpu().numpy())
                val_targets.append(y.detach().cpu().numpy())
                val_frequencies.append(z[:, 0].detach().cpu().numpy())

        val_loss = np.concatenate(val_loss)
        val_pred = np.concatenate(val_pred)
        val_targets = np.concatenate(val_targets)
        val_frequencies = np.concatenate(val_frequencies)

        avg_val_loss = float(np.mean(val_loss))
        avg_val_acc = float(np.mean(val_pred == val_targets))

        old_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]["lr"]

        if new_lr != old_lr:
            print(f"\nLearning rate reduced: {old_lr:.2e} -> {new_lr:.2e}")

        epoch_number = epoch + 1
        completed_epochs = epoch_number
        validation_losses[f"epoch_{epoch_number}"] = val_loss
        validation_predictions[f"epoch_{epoch_number}"] = val_pred
        validation_accuracies[f"epoch_{epoch_number}"] = avg_val_acc

        if avg_val_loss < current_run_best_val_loss:
            current_run_best_val_loss = avg_val_loss
            current_run_best_loss_epoch = epoch_number

        is_new_run_best = avg_val_acc > current_run_best_val_acc
        if is_new_run_best:
            current_run_best_val_acc = avg_val_acc
            current_run_best_acc_epoch = epoch_number
            current_run_best_predictions = val_pred.copy()
            current_run_best_targets = val_targets.copy()
            current_run_best_frequencies = val_frequencies.copy()

            torch.save(
                {
                    "model_state_dict": checkpoint_model.state_dict(),
                    "best_val_acc": current_run_best_val_acc,
                    "best_val_loss": avg_val_loss,
                    "best_epoch": epoch_number,
                    "run_name": run_name,
                    "n_epochs": n_epochs,
                    "learning_rate": optimizer.param_groups[0]["lr"],
                    "weight_decay": weight_decay,
                    "n_base_channels": 40,
                    "hidden_dim": 96,
                    "graph_k": 8,
                    "model_type": "graph_temporal_proto_bit_cnn",
                    "freq_mean": freq_mean,
                    "freq_std": freq_std,
                    "coord_mean": coord_mean.tolist(),
                    "coord_std": coord_std.tolist(),
                    "aux_bit_loss_weight": 0.25,
                    "uses_class_prototypes": True,
                    "checkpoint_metric": "validation_accuracy",
                    "gpu_ids": task1_gpu_ids,
                },
                run_best_model_path,
            )

        if avg_val_acc > all_time_best_val_acc:
            all_time_best_val_acc = avg_val_acc
            all_time_best_val_loss = avg_val_loss

            torch.save(
                {
                    "model_state_dict": checkpoint_model.state_dict(),
                    "best_val_acc": all_time_best_val_acc,
                    "best_val_loss": all_time_best_val_loss,
                    "best_epoch": epoch_number,
                    "run_name": run_name,
                    "n_epochs": n_epochs,
                    "learning_rate": optimizer.param_groups[0]["lr"],
                    "weight_decay": weight_decay,
                    "n_base_channels": 40,
                    "hidden_dim": 96,
                    "graph_k": 8,
                    "model_type": "graph_temporal_proto_bit_cnn",
                    "freq_mean": freq_mean,
                    "freq_std": freq_std,
                    "coord_mean": coord_mean.tolist(),
                    "coord_std": coord_std.tolist(),
                    "aux_bit_loss_weight": 0.25,
                    "uses_class_prototypes": True,
                    "checkpoint_metric": "validation_accuracy",
                    "gpu_ids": task1_gpu_ids,
                },
                best_model_path,
            )

            print(
                f"\nNew all-time best final-task model saved: "
                f"epoch {epoch_number}, val acc {100 * all_time_best_val_acc:.2f}%, "
                f"val loss {all_time_best_val_loss:.4f}"
            )

        history_rows.append({
            "epoch": epoch_number,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_acc_percent": 100 * train_acc,
            "val_loss": avg_val_loss,
            "val_acc": avg_val_acc,
            "val_acc_percent": 100 * avg_val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "is_best_acc_this_run": bool(is_new_run_best),
            "is_best_loss_this_run": bool(epoch_number == current_run_best_loss_epoch),
        })

        summary = {
            "run_name": run_name,
            "status": "running",
            "completed_epochs": completed_epochs,
            "requested_epochs": n_epochs,
            "device": str(device),
            "model_type": "graph_temporal_proto_bit_cnn",
            "freq_mean": freq_mean,
            "freq_std": freq_std,
            "gpu_ids": task1_gpu_ids,
            "uses_data_parallel": isinstance(model, nn.DataParallel),
            "output_dir": str(task1_output_dir),
            "best_model_this_run_path": str(run_best_model_path),
            "all_time_best_model_path": best_model_path,
            "best_val_acc_this_run": current_run_best_val_acc,
            "best_val_acc_percent_this_run": 100 * current_run_best_val_acc,
            "best_acc_epoch_this_run": current_run_best_acc_epoch,
            "best_val_loss_this_run": current_run_best_val_loss,
            "best_loss_epoch_this_run": current_run_best_loss_epoch,
            "all_time_best_val_acc_after_epoch": all_time_best_val_acc,
            "all_time_best_val_acc_percent_after_epoch": 100 * all_time_best_val_acc,
            "all_time_best_val_loss_after_epoch": all_time_best_val_loss,
        }
        write_task1_artifacts(
            task1_output_dir,
            history_rows,
            summary,
            current_run_best_predictions,
            current_run_best_targets,
            current_run_best_frequencies,
        )

        pbar.set_postfix(
            loss_train=f"{train_loss:.4f}",
            acc_train=f"{100 * train_acc:.2f}%",
            loss_val=f"{avg_val_loss:.4f}",
            acc_val=f"{100 * avg_val_acc:.2f}%",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            best_acc=f"{100 * current_run_best_val_acc:.2f}%",
        )
except KeyboardInterrupt:
    interrupted = True
    print("\nTraining interrupted. Writing Task 1 artifacts from completed epochs.")
    raise
finally:
    summary = {
        "run_name": run_name,
        "status": "interrupted" if interrupted else "completed",
        "completed_epochs": completed_epochs,
        "requested_epochs": n_epochs,
        "device": str(device),
        "model_type": "graph_temporal_proto_bit_cnn",
        "freq_mean": freq_mean,
        "freq_std": freq_std,
        "gpu_ids": task1_gpu_ids,
        "uses_data_parallel": isinstance(model, nn.DataParallel),
        "output_dir": str(task1_output_dir),
        "best_model_this_run_path": str(run_best_model_path),
        "all_time_best_model_path": best_model_path,
        "best_val_acc_this_run": current_run_best_val_acc,
        "best_val_acc_percent_this_run": 100 * current_run_best_val_acc,
        "best_acc_epoch_this_run": current_run_best_acc_epoch,
        "best_val_loss_this_run": current_run_best_val_loss if current_run_best_val_loss < float("inf") else None,
        "best_loss_epoch_this_run": current_run_best_loss_epoch,
        "all_time_best_val_acc_after_run": all_time_best_val_acc,
        "all_time_best_val_acc_percent_after_run": 100 * all_time_best_val_acc,
        "all_time_best_val_loss_after_run": all_time_best_val_loss if all_time_best_val_loss < float("inf") else None,
    }
    write_task1_artifacts(
        task1_output_dir,
        history_rows,
        summary,
        current_run_best_predictions,
        current_run_best_targets,
        current_run_best_frequencies,
    )

print(f"\nBest validation accuracy in this final-task run: {100 * current_run_best_val_acc:.2f}%")
print(f"Best accuracy epoch in this final-task run: {current_run_best_acc_epoch}")
print(f"Best validation loss in this final-task run: {current_run_best_val_loss:.4f}")
print(f"Best loss epoch in this final-task run: {current_run_best_loss_epoch}")
print(f"All-time best final-task validation accuracy after this run: {100 * all_time_best_val_acc:.2f}%")
print(f"Task 1 output directory: {task1_output_dir}")
print(f"Best final-task model path: {best_model_path}")

Visible GPU status from utilization samples:
 - logical GPU 0 / physical GPU 0: util_samples=[29, 30, 24, 36, 40], max_util=40%, free_mem=9291/11264 MiB, used_mem=1719 MiB -> busy
 - logical GPU 1 / physical GPU 1: util_samples=[29, 28, 20, 41, 40], max_util=41%, free_mem=9577/11264 MiB, used_mem=1433 MiB -> busy
 - logical GPU 2 / physical GPU 2: util_samples=[0, 0, 0, 0, 0], max_util=0%, free_mem=11007/11264 MiB, used_mem=3 MiB -> free
 - logical GPU 3 / physical GPU 3: util_samples=[0, 0, 0, 0, 0], max_util=0%, free_mem=11007/11264 MiB, used_mem=3 MiB -> free
 - logical GPU 4 / physical GPU 4: util_samples=[0, 0, 0, 0, 0], max_util=0%, free_mem=11007/11264 MiB, used_mem=3 MiB -> free
 - logical GPU 5 / physical GPU 5: util_samples=[0, 0, 0, 0, 0], max_util=0%, free_mem=11007/11264 MiB, used_mem=3 MiB -> free
 - logical GPU 6 / physical GPU 6: util_samples=[0, 0, 0, 0, 0], max_util=0%, free_mem=11007/11264 MiB, used_mem=3 MiB -> free
 - logical GPU 7 / physical GPU 7: util_samples=[0

  0%|          | 0/120 [00:00<?, ?it/s]


New all-time best final-task model saved: epoch 1, val acc 39.42%, val loss 1.6313


  2%|▏         | 2/120 [00:56<54:20, 27.63s/it, acc_train=39.99%, acc_val=40.84%, best_acc=40.84%, loss_train=1.7158, loss_val=1.5718, lr=8.00e-04]  


New all-time best final-task model saved: epoch 2, val acc 40.84%, val loss 1.5718


...

 12%|█▏        | 14/120 [05:49<42:52, 24.27s/it, acc_train=63.91%, acc_val=41.79%, best_acc=43.29%, loss_train=1.2138, loss_val=1.6619, lr=4.00e-04]


Learning rate reduced: 8.00e-04 -> 4.00e-04


 16%|█▌        | 19/120 [07:50<40:41, 24.17s/it, acc_train=75.52%, acc_val=40.15%, best_acc=43.29%, loss_train=0.9669, loss_val=1.7745, lr=4.00e-04]


Training interrupted. Writing Task 1 artifacts from completed epochs.


KeyboardInterrupt: 

In [ ]:
# 4. Benchmarking

In [ ]:
# Benchmarking: best-accuracy epoch from the most recent final-task run

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if "validation_losses" not in globals() or len(validation_losses) == 0:
    raise RuntimeError("No validation_losses found. Run the training cell first.")

if "validation_predictions" not in globals() or len(validation_predictions) == 0:
    raise RuntimeError("No validation_predictions found. Run the training cell first.")

if "validation_accuracies" not in globals() or len(validation_accuracies) == 0:
    raise RuntimeError("No validation_accuracies found. Run the training cell first.")

val_keys = sorted(validation_losses.keys(), key=lambda x: int(x.split("_")[1]))
val_epochs = [int(k.split("_")[1]) for k in val_keys]
epoch_losses = [np.mean(validation_losses[k]) for k in val_keys]
epoch_accuracies = [100 * validation_accuracies[k] for k in val_keys]

best_epoch_key = max(validation_accuracies.keys(), key=lambda k: validation_accuracies[k])
best_epoch = int(best_epoch_key.split("_")[1])
best_val_loss = np.mean(validation_losses[best_epoch_key])
best_val_acc = 100 * validation_accuracies[best_epoch_key]

print("Benchmarking recent final-task run")
print(f"Best validation-accuracy epoch in recent run: {best_epoch}")
print(f"Validation loss at that epoch: {best_val_loss:.4f}")
print(f"Overall validation accuracy using best-accuracy epoch from recent final-task run: {best_val_acc:.2f}%")

predictions = np.array(validation_predictions[best_epoch_key])
true_labels = val_true_labels if "val_true_labels" in globals() else Zval[:, 1].astype(int)
accuracy = predictions == true_labels

artifact_dir = Path(task1_output_dir) if "task1_output_dir" in globals() else None
if artifact_dir is not None:
    pd.DataFrame({
        "epoch": val_epochs,
        "val_loss": epoch_losses,
        "val_acc_percent": epoch_accuracies,
    }).to_csv(artifact_dir / "benchmark_recent_history.csv", index=False)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(val_epochs, epoch_losses, marker="o", markersize=3, label="Loss")
ax1.axvline(best_epoch, linestyle="--", label=f"Best acc epoch: {best_epoch}")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Mean loss")
ax2 = ax1.twinx()
ax2.plot(val_epochs, epoch_accuracies, marker="s", markersize=3, color="tab:green", label="Accuracy")
ax2.set_ylabel("Accuracy [%]")
ax1.set_title("Final task: validation loss and accuracy, recent run")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="best")
fig.tight_layout()
if artifact_dir is not None:
    fig.savefig(artifact_dir / "validation_loss_accuracy_recent.png", dpi=150)
plt.show()

frequencies = Zval[:, 0]
unique_freqs = np.unique(frequencies)
frequency_accuracies = np.array([
    accuracy[frequencies == f].mean() * 100
    for f in unique_freqs
])
frequency_df = pd.DataFrame({"frequency": unique_freqs, "accuracy_percent": frequency_accuracies})
if artifact_dir is not None:
    frequency_df.to_csv(artifact_dir / "benchmark_recent_accuracy_by_frequency.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(unique_freqs, frequency_accuracies, marker="o", markersize=3)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Accuracy [%]")
ax.set_title("Final task: accuracy per frequency, recent run")
fig.tight_layout()
if artifact_dir is not None:
    fig.savefig(artifact_dir / "accuracy_by_frequency_recent.png", dpi=150)
plt.show()

patterns = true_labels
unique_patterns = np.unique(patterns)
pattern_accuracies = np.array([
    accuracy[patterns == p].mean() * 100
    for p in unique_patterns
])
pattern_df = pd.DataFrame({"pattern": unique_patterns.astype(int), "accuracy_percent": pattern_accuracies})
if artifact_dir is not None:
    pattern_df.to_csv(artifact_dir / "benchmark_recent_accuracy_by_pattern.csv", index=False)

x = np.arange(len(unique_patterns))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x, pattern_accuracies)
ax.set_xticks(x)
ax.set_xticklabels(unique_patterns.astype(int).astype(str))
ax.set_xlabel("Pattern")
ax.set_ylabel("Accuracy [%]")
ax.set_title("Final task: accuracy per pattern, recent run")
fig.tight_layout()
if artifact_dir is not None:
    fig.savefig(artifact_dir / "accuracy_by_pattern_recent.png", dpi=150)
plt.show()

In [ ]:
# Benchmarking: all-time best final-task model selected by validation accuracy

import os
import getpass
import subprocess
import torch
import numpy as np
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print(f"Benchmarking all-time best final-task model on {device}.")

best_model_path = f"/home/{getpass.getuser()}/best_model_final_task_1_graph_proto_bits.pth"

if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"No saved best final-task model found at: {best_model_path}")

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    load_target = model.module if isinstance(model, nn.DataParallel) else model
    load_target.load_state_dict(checkpoint["model_state_dict"])

    print("Loaded all-time best final-task checkpoint")
    print("Best validation accuracy:", f"{100 * checkpoint.get('best_val_acc', 0.0):.2f}%")
    print("Best validation loss:", checkpoint.get("best_val_loss", "unknown"))
    print("Best epoch:", checkpoint.get("best_epoch", "unknown"))
    print("Run name:", checkpoint.get("run_name", "unknown"))
else:
    load_target = model.module if isinstance(model, nn.DataParallel) else model
    load_target.load_state_dict(checkpoint)
    print("Loaded old-format final-task model state_dict.")
    print("Warning: no best_val_acc metadata available.")

model.to(device)
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y, y_bits, z in val_loader:
        x = x.to(device)
        ckpt_freq_mean = checkpoint.get("freq_mean", freq_mean if "freq_mean" in globals() else 0.0) if isinstance(checkpoint, dict) else (freq_mean if "freq_mean" in globals() else 0.0)
        ckpt_freq_std = checkpoint.get("freq_std", freq_std if "freq_std" in globals() else 1.0) if isinstance(checkpoint, dict) else (freq_std if "freq_std" in globals() else 1.0)
        freq = ((z[:, 0].to(device) - ckpt_freq_mean) / ckpt_freq_std).float()

        pred, bit_pred = model(x, freq)
        pred_class = torch.argmax(pred, dim=-1)

        all_preds.append(pred_class.detach().cpu().numpy())
        all_targets.append(y.numpy())

all_preds = np.concatenate(all_preds)
true_labels = np.concatenate(all_targets)
accuracy = all_preds == true_labels
overall_acc = accuracy.mean() * 100

print(f"Overall validation accuracy using all-time best final-task model: {overall_acc:.2f}%")

# Accuracy per frequency
frequencies = Zval[:, 0]
unique_freqs = np.unique(frequencies)

frequency_accuracies = np.array([
    accuracy[frequencies == f].mean() * 100
    for f in unique_freqs
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(unique_freqs, frequency_accuracies, marker="o", markersize=3)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Accuracy [%]")
ax.set_title("Final task: accuracy per frequency, all-time best")
fig.tight_layout()
plt.show()

# Accuracy per pattern across all frequencies
patterns = true_labels
unique_patterns = np.unique(patterns)

pattern_accuracies = np.array([
    accuracy[patterns == p].mean() * 100
    for p in unique_patterns
])

x = np.arange(len(unique_patterns))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x, pattern_accuracies)
ax.set_xticks(x)
ax.set_xticklabels(unique_patterns.astype(int).astype(str))
ax.set_xlabel("Pattern")
ax.set_ylabel("Accuracy [%]")
ax.set_title("Final task: accuracy per pattern, all-time best")
fig.tight_layout()
plt.show()